# Support Ticket Classification and Prioritization

This notebook builds an ML system to automatically classify and prioritize customer support tickets using Natural Language Processing (NLP) and Machine Learning.

## Objective
- Classify tickets into categories (e.g., Fileservice, Software, etc.)
- Assign priority levels (High, Medium, Low)
- Improve support operations by automating ticket sorting

## 1. Import Libraries and Setup

In [ ]:

pip install pandas numpy matplotlib seaborn scikit-learn nltk joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

ModuleNotFoundError: No module named 'pandas'

## 2. Load Support Ticket Dataset

In [ ]:
# Load the dataset
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

# Merge X and y for train and test
train_df = pd.merge(X_train, y_train, on='id')
test_df = pd.merge(X_test, y_test, on='id')

# Combine for full dataset
df = pd.concat([train_df, test_df], ignore_index=True)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample data:")
print(df.head())

NameError: name 'pd' is not defined

## 3. Explore and Visualize Ticket Data

In [ ]:
# Basic exploration
print("Category distribution:")
print(df['category_truth'].value_counts())

# Visualize category distribution
plt.figure(figsize=(10, 6))
df['category_truth'].value_counts().plot(kind='bar')
plt.title('Distribution of Ticket Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# Text length analysis
df['text_length'] = df['text'].apply(len)
print("\nText length statistics:")
print(df['text_length'].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df['text_length'], bins=50)
plt.title('Distribution of Ticket Text Length')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

## 4. Assign Priority Levels
Since the dataset doesn't have priority, we'll assign based on categories:
- High: Software, Computer-Services (technical issues requiring immediate attention)
- Medium: Fileservice, O365 (access and productivity issues)
- Low: Support general, Active Directory, EOL (general inquiries, maintenance)

In [ ]:
# Define priority mapping
priority_map = {
    'Software': 'High',
    'Computer-Services': 'High',
    'Fileservice': 'Medium',
    'O365': 'Medium',
    'Support general': 'Low',
    'Active Directory': 'Low',
    'EOL': 'Low'
}

df['priority'] = df['category_truth'].map(priority_map)

print("Priority distribution:")
print(df['priority'].value_counts())

# Visualize priority
plt.figure(figsize=(6, 4))
df['priority'].value_counts().plot(kind='bar')
plt.title('Distribution of Ticket Priorities')
plt.xlabel('Priority')
plt.ylabel('Count')
plt.show()

## 5. Text Cleaning and Preprocessing

In [ ]:
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

def preprocess_text(text):
    # Clean text
    text = clean_text(text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_text)

print("Sample cleaned text:")
print(df[['text', 'cleaned_text']].head())

## 6. Convert Text to Numerical Features

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = tfidf.fit_transform(df['cleaned_text'])

# Labels
y_category = df['category_truth']
y_priority = df['priority']

# Split into train and test (using the original split)
X_train_vec = tfidf.transform(train_df['text'].apply(preprocess_text))
X_test_vec = tfidf.transform(test_df['text'].apply(preprocess_text))
y_train_cat = train_df['category_truth']
y_test_cat = test_df['category_truth']

# For priority
train_df['priority'] = train_df['category_truth'].map(priority_map)
test_df['priority'] = test_df['category_truth'].map(priority_map)
y_train_pri = train_df['priority']
y_test_pri = test_df['priority']

print("TF-IDF shape:", X.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

## 7. Train Category Classification Model

In [ ]:
# Train Logistic Regression for categories
cat_model = LogisticRegression(random_state=42, max_iter=1000)
cat_model.fit(X_train_vec, y_train_cat)

# Predictions
y_pred_cat = cat_model.predict(X_test_vec)

print("Category Classification Report:")
print(classification_report(y_test_cat, y_pred_cat))

# Confusion Matrix
cm_cat = confusion_matrix(y_test_cat, y_pred_cat)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_cat, annot=True, fmt='d', xticklabels=cat_model.classes_, yticklabels=cat_model.classes_)
plt.title('Confusion Matrix - Category Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 8. Train Priority Prediction Model

In [ ]:
# Train Naive Bayes for priority
pri_model = MultinomialNB()
pri_model.fit(X_train_vec, y_train_pri)

# Predictions
y_pred_pri = pri_model.predict(X_test_vec)

print("Priority Classification Report:")
print(classification_report(y_test_pri, y_pred_pri))

# Confusion Matrix
cm_pri = confusion_matrix(y_test_pri, y_pred_pri)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_pri, annot=True, fmt='d', xticklabels=pri_model.classes_, yticklabels=pri_model.classes_)
plt.title('Confusion Matrix - Priority Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 9. Save Models and Inference Pipeline

In [ ]:
import joblib

# Save models and vectorizer
joblib.dump(cat_model, 'category_model.pkl')
joblib.dump(pri_model, 'priority_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Models saved successfully!")

# Inference function
def classify_ticket(ticket_text):
    # Preprocess
    cleaned = preprocess_text(ticket_text)
    # Vectorize
    vec = tfidf.transform([cleaned])
    # Predict
    category = cat_model.predict(vec)[0]
    priority = pri_model.predict(vec)[0]
    return category, priority

# Test inference
sample_ticket = "Java installation failed on my computer"
cat, pri = classify_ticket(sample_ticket)
print(f"Sample ticket: '{sample_ticket}'")
print(f"Predicted Category: {cat}")
print(f"Predicted Priority: {pri}")

## Summary and Insights

This ML system automates support ticket classification and prioritization:

- **Category Classification**: Uses TF-IDF features and Logistic Regression to classify tickets into 7 categories with high accuracy.
- **Priority Assignment**: Based on business logic mapping categories to High/Medium/Low priority levels.
- **Benefits**: Reduces manual sorting time, ensures urgent issues are addressed quickly, improves response times and customer satisfaction.

The system can be integrated into support ticketing systems for real-time classification of incoming tickets.